# Task 2 — Profiling and Cleaning

All active sources are cleaned through the shared functions in `src/`. Missing required values are **not** dropped here; they remain for Task 3 so the rejection report is complete. Partial dates are not completed with invented month/day values.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from main import build_kaust, build_kfupm, build_ksu
from src import profile
from src.config import get_paths, load_config, year_range

config = load_config(ROOT)
paths = get_paths(config).ensure()
min_year, max_year = year_range(config)


In [ ]:
kaust = build_kaust(config, paths, min_year, max_year)
kfupm = build_kfupm(config, paths, min_year, max_year)
ksu = build_ksu(config, paths, min_year, max_year)

print("KAUST cleaned:", len(kaust))
print("KFUPM cleaned:", len(kfupm))
print("KSU cleaned:", len(ksu))


## Missing-value profile

In [ ]:
for name, frame in [("KAUST", kaust), ("KFUPM", kfupm), ("KSU", ksu)]:
    print(f"\n{name}")
    display(profile.missing_report(frame))


## Duplicate and year checks

In [ ]:
for name, frame in [("KAUST", kaust), ("KFUPM", kfupm), ("KSU", ksu)]:
    print(f"\n{name}")
    display(profile.duplicate_report(frame, [["research_id"], ["doi"]]))
    display(profile.year_distribution(frame))


### Cleaning decisions

- Mandatory-field failures are retained for validation rather than silently removed.
- `publication_date` is populated only when the source supplies year, month, and day.
- DOI values are normalized to bare lowercase form.
- KFUPM uses the Pure organisational-unit field for the Computer Engineering department filter.
- KSU `publication_year` comes from the annual source file and its `url` is the dataset URL because no per-publication URL is supplied.
- PNU is excluded from the final dataset; its notebooks are preserved under `notebooks/excluded_pnu/` for audit.